# Summit-Sim: Complete E2E Workflow Demo

Demonstrates the full Summit-Sim workflow:
1. **Phase 0: Teacher Review** - Teacher generates and approves scenario
2. **Phase 1: Scenario Generation** - AI generates wilderness rescue scenario
3. **Phase 2: Simulation** - Student navigates scenario with human-in-the-loop
4. **Phase 3: Debrief** - Performance analysis and learning report

All phases are traced to MLflow for visibility.

## 1. Setup & Imports

In [13]:
import warnings

# Suppress autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.graphs.teacher_review import create_teacher_review_graph
from summit_sim.schemas import TeacherConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import enable_tracing, summit_session

# Initialize MLflow tracing
enable_tracing()

print(f"✓ MLflow: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key: {bool(settings.openrouter_api_key)}")

✓ MLflow: https://mlflow.bhamm-lab.com
✓ Experiment: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key: True


## 2. Phase 0: Teacher Review Flow

Demonstrates the teacher workflow:
1. **Configuration** - Teacher sets scenario parameters
2. **Generation** - AI creates scenario from config
3. **Review** - Teacher reviews scenario via interrupt
4. **Approval** - Teacher approves scenario
5. **Share** - Unique URL generated for students

In [14]:
# Teacher configuration
teacher_config = TeacherConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Teacher Config: {teacher_config.activity_type} scenario for {teacher_config.num_participants} participants"
)
print(f"Difficulty: {teacher_config.difficulty}")
print(f"Class ID: {teacher_config.class_id}")

Teacher Config: hiking scenario for 3 participants
Difficulty: med
Class ID: demo-class-2024


In [15]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create teacher review graph with memory
memory = InMemorySaver()
teacher_graph = create_teacher_review_graph(checkpointer=memory)

# Initialize state
initial_teacher_state = {
    "teacher_config": teacher_config,
    "scenario_draft": None,
    "scenario_id": "",
    "class_id": "",
    "retry_count": 0,
    "feedback_history": [],
    "approval_status": None,
}

print("✓ Teacher review graph initialized")

✓ Teacher review graph initialized


In [16]:
# Run teacher review graph with MLflow session tracking
import mlflow

# Generate a scenario ID for this session
scenario_id = generate_scenario_id()

with summit_session(teacher_config, scenario_id=scenario_id, phase="gen") as (
    session_id,
    graph_config,
):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting teacher review...\n")
    print("=" * 60)
    print("TEACHER REVIEW FLOW")
    print("=" * 60 + "\n")

    # Run to interrupt point (scenario generation + review)
    result = await teacher_graph.ainvoke(initial_teacher_state, graph_config)

    # Display generated scenario for review
    scenario = result["scenario_draft"]
    print(f"\n📋 Generated Scenario: {scenario.title}")
    print(f"Setting: {scenario.setting}")
    print(f"Patient: {scenario.patient_summary}")
    print(f"Turns: {len(scenario.turns)}")
    print("\nLearning Objectives:")
    for obj in scenario.learning_objectives:
        print(f"  • {obj}")

    # Simulate teacher approval
    print("\n" + "-" * 60)
    print("Teacher reviews scenario and clicks 'Approve'...")
    print("-" * 60 + "\n")

    # Resume with approval
    final_result = await teacher_graph.ainvoke(
        Command(resume={"decision": "approve"}), graph_config
    )

    # Display shareable URL
    class_id = final_result["class_id"]
    shareable_url = f"/scenario/{scenario_id}?class_id={class_id}"

    print("\n" + "=" * 60)
    print("✅ SCENARIO APPROVED")
    print("=" * 60)
    print(f"\n📍 Scenario ID: {scenario_id}")
    print(f"👥 Class ID: {class_id}")
    print(f"\n🔗 Shareable URL: {shareable_url}")
    print("\n✓ Ready for students to join!")

    # Store for next phase
    approved_scenario = scenario
    approved_scenario_id = scenario_id
    approved_class_id = class_id

2026/03/24 11:24:53 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd2829c0> was created in a different Context



📋 Session ID: 936a38ba-cca9-47b3-8a46-c5812b89c1ae
🎮 Starting teacher review...

TEACHER REVIEW FLOW



2026/03/24 11:25:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd283380> was created in a different Context
2026/03/24 11:25:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd3cd0c0> was created in a different Context
2026/03/24 11:25:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29ddc73fc0> was created in a different Context
2026/03/24 11:25:18 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.TeacherConfig from checkpoint. This will be block


📋 Generated Scenario: Hiker Down: The Ridge Trail
Setting: Alpine ridge trail at 8,500ft, late afternoon. Windy, temperature falling rapidly, light rain beginning to fall. Rough, uneven terrain.
Patient: 30-year-old female, experienced hiker. Previously talkative, now notably quiet, stumbling, difficulty manipulating small objects (like gear buckles). Shivering uncontrollably. Skin is cool to the touch.
Turns: 4

Learning Objectives:
  • Recognition of hypothermia symptoms in a wilderness setting.
  • Prioritizing passive rewarming and energy conservation over immediate movement.
  • Assessing patient stability for evacuation vs. continued rescue efforts.

------------------------------------------------------------
Teacher reviews scenario and clicks 'Approve'...
------------------------------------------------------------


✅ SCENARIO APPROVED

📍 Scenario ID: scn-d592ba69
👥 Class ID: ffa98b

🔗 Shareable URL: /scenario/scn-d592ba69?class_id=ffa98b

✓ Ready for students to join!
🏃 Vie

Trace(trace_id=tr-62dfc127d36c6c39dd47cf2902552498)

## 3. Phase 1: Scenario Generation (Approved)

The approved scenario from Phase 0 is now ready for students.

In [17]:
# Use the approved scenario from Phase 0
scenario = approved_scenario
scenario_id = approved_scenario_id
config = teacher_config

print(f"Using approved scenario: {scenario.title}")
print(f"Scenario ID: {scenario_id}")
print(f"Class ID: {approved_class_id}")
print("\n✓ Ready for Phase 2: Simulation")

Using approved scenario: Hiker Down: The Ridge Trail
Scenario ID: scn-d592ba69
Class ID: ffa98b

✓ Ready for Phase 2: Simulation


## 4. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [18]:
# Configuration
config = TeacherConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Generating: {config.activity_type} scenario for {config.num_participants} participants"
)
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Scenario ID: {scenario_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating: hiking scenario for 3 participants
------------------------------------------------------------
✓ Title: Ridge Line Fracture
✓ Setting: A narrow, rocky hiking trail on a high ridge during late afternoon. Temperature is dropping, and wind speeds are increasing. The location is approximately 3 miles from the trailhead.
✓ Patient: 28-year-old female hiker. Mentally alert but in significant distress. Deformity present in the lower right leg, near the ankle, with localized swelling. Skin is pale and cool.
✓ Turns: 5
✓ Scenario ID: scn-9ad144b3

Learning Objectives:
  • Performing a thorough primary survey in a wilderness setting
  • Correct assessment and improvisation of limb immobilization/splinting
  • Managing a patient with a severe orthopedic injury to prevent secondary complications (shock/worsening trauma)


Trace(trace_id=tr-94056480b887785ff2684f9c410ed8b4)

### Display Scenario Structure

In [19]:
for turn in scenario.turns:
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"\n{'=' * 60}")
    print(f"Turn {turn.turn_id} {marker}")
    print(f"{'=' * 60}")
    print(
        turn.narrative_text[:200] + "..."
        if len(turn.narrative_text) > 200
        else turn.narrative_text
    )
    print("\nChoices:")
    for choice in turn.choices:
        mark = "✓" if choice.is_correct else " "
        next_turn = (
            "END" if choice.next_turn_id is None else f"Turn {choice.next_turn_id}"
        )
        print(f"  [{mark}] {choice.description[:60]}... → {next_turn}")


Turn 0 (START)
You are hiking on a ridgeline trail with two partners. A 28-year-old female hiker in your party slips on a loose rock and falls about 10 feet down a rocky embankment. You reach her quickly. She is con...

Choices:
  [✓] Immediately stabilize the head/neck, perform a trauma assess... → Turn 1
  [ ] Have the two other hikers assist the patient to stand up imm... → Turn 2

Turn 1 
You have stabilized her head/neck and performed a primary assessment. Her airway is clear, breathing is stable but rapid due to pain, and circulation is intact but the foot is cold and pale, indicatin...

Choices:
  [✓] Pad around the ankle with spare clothing, use hiking poles a... → Turn 3
  [ ] Apply a tourniquet high and tight on the thigh immediately t... → Turn 4

Turn 2 
As the patient attempts to stand, she screams in agony and collapses again. Her pulse is now weak and rapid, and she is becoming lethargic. The movement has clearly worsened her injury and contributed...

Choices:
  [✓] St

## 3. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [20]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create graph with memory
memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
    "debrief_report": None,
}

print("✓ Graph initialized")
print(f"✓ Starting at turn {initial_state['current_turn_id']}")

✓ Graph initialized
✓ Starting at turn 0


In [21]:
# Run simulation with MLflow session tracking

from summit_sim.tracing import log_simulation_results, summit_session

with summit_session(config, scenario_id=scenario_id, phase="sim") as (
    session_id,
    graph_config,
):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting simulation...\n")
    print("=" * 60)
    print("INTERACTIVE SIMULATION")
    print("=" * 60 + "\n")

    # Link traces to session using metadata
    mlflow.update_current_trace(
        metadata={
            "mlflow.trace.session": session_id,
            "mlflow.trace.user": "student",
            "scenario_id": scenario_id,
        }
    )

    current_state = initial_state
    turn_count = 0
    simulation_complete = False

    print("\n=== STARTING SIMULATION ===\n")

    while not simulation_complete:
        input_state = initial_state if turn_count == 0 else None
        turn_count += 1
        print(f"--- TURN {turn_count} ---")

        async for event in graph.astream(input_state, graph_config):
            if "__interrupt__" in event:
                interrupt_obj = event["__interrupt__"]
                if hasattr(interrupt_obj, "value"):
                    interrupt_data = interrupt_obj.value
                elif isinstance(interrupt_obj, (list, tuple)):
                    interrupt_data = (
                        interrupt_obj[0].value
                        if hasattr(interrupt_obj[0], "value")
                        else interrupt_obj[0]
                    )
                else:
                    interrupt_data = interrupt_obj

                print(f"\n📖 {interrupt_data['narrative']}\n")
                print("Choices:")
                for i, choice in enumerate(interrupt_data["choices"], 1):
                    print(f"  {i}. {choice['description']}")

                while True:
                    try:
                        sel = int(input("\nSelect: ")) - 1
                        if 0 <= sel < len(interrupt_data["choices"]):
                            break
                        print("Invalid")
                    except ValueError:
                        print("Enter number")

                selected = interrupt_data["choices"][sel]
                current_state = await graph.ainvoke(
                    Command(resume={"choice_id": selected["choice_id"]}), graph_config
                )
                break

            # Check for graph completion
            if event == {}:
                simulation_complete = True
                break

        # Check if state indicates completion
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("Safety stop")
            break

    print("\n=== SIMULATION COMPLETE ===")

    # Log final results to MLflow parent run
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
        debrief_report=current_state["debrief_report"],
    )
    print("\n✓ Results logged to MLflow")
    print(f"✓ Total turns: {len(current_state['transcript'])}")
    print(f"✓ Learning moments: {len(current_state['key_learning_moments'])}")

2026/03/24 11:25:45 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.
2026/03/24 11:25:45 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd2f5880> was created in a different Context



📋 Session ID: 1e6f3140-5d5c-4571-811e-05470734fa85
🎮 Starting simulation...

INTERACTIVE SIMULATION


=== STARTING SIMULATION ===

--- TURN 1 ---


2026/03/24 11:25:45 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd2bb180> was created in a different Context



📖 You are hiking on a ridgeline trail with two partners. A 28-year-old female hiker in your party slips on a loose rock and falls about 10 feet down a rocky embankment. You reach her quickly. She is conscious, screaming in pain, and holding her right lower leg, which shows an obvious, severe deformity near the ankle. She is pale, and the ambient temperature is dropping rapidly. What is your immediate action?

Choices:
  1. Immediately stabilize the head/neck, perform a trauma assessment, and immobilize the injured leg in place.
  2. Have the two other hikers assist the patient to stand up immediately to begin slowly walking back to the trailhead.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/24 11:25:54 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd2b8a40> was created in a different Context
2026/03/24 11:26:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd2f7080> was created in a different Context
2026/03/24 11:26:02 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd3fa080> was created in a different Context
2026/03/24 11:26:02 WARNING mlflow.utils.autologging_utils: Encountered un

...

--- TURN 3 ---

📖 Due to the suboptimal handling of the injury and the patient's deteriorating condition, a successful self-evacuation is no longer possible. The injury has likely resulted in severe soft tissue damage due to improper stabilization or forced movement, and the patient is now in compensated shock. Help is required immediately. Simulation ends here.

Choices:
  1. End simulation. Your actions have contributed to a dangerous delay and potential permanent injury.
  2. End simulation. Continued attempts to move the patient have led to total collapse.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/24 11:26:18 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd147fc0> was created in a different Context
2026/03/24 11:26:22 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


=== SIMULATION COMPLETE ===

✓ Results logged to MLflow
✓ Total turns: 3
✓ Learning moments: 6
🏃 View run sim-demo-class-2024-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/d885cb51f2274e7e8335f653387ff1c1
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-e497f69d4fd7813468fdf4187da6483d), Trace(trace_id=tr-a056867ff18ae1bd15aa678e0740f729)]

2026/03/24 11:26:30 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f2a21c0b560> at 0x7f29dd30f500> was created in a different Context


## 4. Phase 3: Debrief

View the comprehensive performance report generated automatically by the simulation graph.

In [22]:
debrief_report = current_state["debrief_report"]

print("=" * 60)
print("DEBRIEF REPORT")
print("=" * 60)

print(f"\n📊 Final Score: {debrief_report.final_score:.1f}%")
status_emoji = "✅" if debrief_report.completion_status == "pass" else "❌"
print(f"{status_emoji} Status: {debrief_report.completion_status.upper()}")

print("\n📝 Summary:")
print(debrief_report.summary)

if debrief_report.key_mistakes:
    print("\n⚠️ Key Mistakes:")
    for mistake in debrief_report.key_mistakes:
        print(f"  • {mistake}")

if debrief_report.strong_actions:
    print("\n💪 Strong Actions:")
    for action in debrief_report.strong_actions:
        print(f"  • {action}")

if debrief_report.teaching_points:
    print("\n📚 Teaching Points:")
    for point in debrief_report.teaching_points:
        print(f"  • {point}")

if debrief_report.best_next_actions:
    print("\n🎯 Recommendations:")
    for rec in debrief_report.best_next_actions:
        print(f"  • {rec}")

DEBRIEF REPORT

📊 Final Score: 66.7%
❌ Status: FAIL

📝 Summary:
The student demonstrated a strong initial grasp of wilderness trauma assessment protocols, correctly prioritizing C-spine stabilization. However, the simulation performance suffered due to a critical error in clinical judgment regarding hemorrhage control. The application of a tourniquet for a standard fracture is a major procedural error that carries a high risk of preventable tissue damage. Ultimately, the student showed professional maturity by recognizing that their actions had deteriorated the situation, though the overall performance did not meet the simulation threshold.

⚠️ Key Mistakes:
  • Applied a tourniquet inappropriately to a closed orthopedic fracture.
  • Failed to differentiate between a life-threatening arterial bleed and a deformity requiring immobilization.

💪 Strong Actions:
  • Prioritized C-spine stabilization correctly during the primary survey.
  • Recognized the need to stop and reassess when the

Trace(trace_id=tr-5c65d9d2c41d2bbc4603da67327a8421)

## 5. Results Summary

Complete transcript and learning moments from the simulation.

In [23]:
print("\n" + "=" * 60)
print("COMPLETE TRANSCRIPT")
print("=" * 60 + "\n")

for i, entry in enumerate(current_state["transcript"], 1):
    status = "✓" if entry["was_correct"] else "✗"
    print(f"Turn {i} {status}")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print()


COMPLETE TRANSCRIPT

Turn 1 ✓
  Situation: You are hiking on a ridgeline trail with two partners. A 28-year-old female hike...
  Choice: Immediately stabilize the head/neck, perform a trauma assessment, and immobilize the injured leg in place.
  Feedback: Excellent choice. By prioritizing C-spine stabilization and performing a thorough trauma assessment,...

Turn 2 ✗
  Situation: You have stabilized her head/neck and performed a primary assessment. Her airway...
  Choice: Apply a tourniquet high and tight on the thigh immediately to prevent possible internal bleeding.
  Feedback: Applying a tourniquet in this case is inappropriate and potentially harmful. Tourniquets are strictl...

Turn 3 ✓
  Situation: Due to the suboptimal handling of the injury and the patient's deteriorating con...
  Choice: End simulation. Your actions have contributed to a dangerous delay and potential permanent injury.
  Feedback: While this outcome is difficult, acknowledging that your management failed to st

In [24]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{i}. {moment}")


KEY LEARNING MOMENTS

1. Always maintain C-spine stabilization following a significant fall until a formal assessment can rule out spinal injury.
2. Conduct a rapid primary survey (ABCDE) to ensure hemodynamic stability before dedicating resources exclusively to an extremity injury.
3. Tourniquets should only be used for life-threatening, uncontrollable arterial bleeding, not for orthopedic injuries.
4. When a distal pulse is absent or weak, the primary goal is to splint and potentially realign (if trained/permitted) to alleviate pressure on nerves and vessels, rather than restricting all blood flow to the limb.
5. Identifying neurovascular status—specifically pulses, motor function, and sensation—is mandatory to detect and monitor for permanent tissue damage.
6. Prioritize early, proactive immobilization; force-moving a patient with a comminuted fracture significantly increases the risk of secondary trauma and shock.


---
## ✅ Complete Workflow Demo

All phases executed:
- ✓ Scenario generated with AI
- ✓ Simulation completed with human-in-the-loop
- ✓ Debrief report generated
- ✓ All traces logged to MLflow

View traces in MLflow UI at your configured tracking URI.